In [ ]:
import numpy as np
import pandas as pd
import json 
import ast


In [2]:
df = pd.read_csv('SGJobData.CSV')
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048585 entries, 0 to 1048584
Data columns (total 22 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   categories                          1044597 non-null  object 
 1   employmentTypes                     1044597 non-null  object 
 2   metadata_expiryDate                 1044597 non-null  object 
 3   metadata_isPostedOnBehalf           1048585 non-null  bool   
 4   metadata_jobPostId                  1044597 non-null  object 
 5   metadata_newPostingDate             1044597 non-null  object 
 6   metadata_originalPostingDate        1044597 non-null  object 
 7   metadata_repostCount                1048585 non-null  int64  
 8   metadata_totalNumberJobApplication  1048585 non-null  int64  
 9   metadata_totalNumberOfView          1048585 non-null  int64  
 10  minimumYearsExperience              1048585 non-null  int64  
 11  numberOfVac

In [3]:
#df.iloc[0:50000].to_excel('first 50k.xlsx')

# Cleaning up steps #
1) Remove truly blank roles
3) Remove duplicates if any.

# Remove truly blank rows #

In [4]:
# Remove truly blank rows #

categories_blank = df['categories'].isna() | (df['categories'].str.strip() == '')
employmentTypes_blank = df['employmentTypes'].isna() | (df['employmentTypes'].str.strip() == '')

print("categories blank count:", categories_blank.sum())
print("employmentTypes blank count:", employmentTypes_blank.sum())
print("both blank:", (categories_blank & employmentTypes_blank).sum())
print("categories blank but employmentTypes not:", (categories_blank & ~employmentTypes_blank).sum())
print("employmentTypes blank but categories not:", (~categories_blank & employmentTypes_blank).sum())
print("same rows (masks identical):", categories_blank.equals(employmentTypes_blank))


categories blank count: 3988
employmentTypes blank count: 3988
both blank: 3988
categories blank but employmentTypes not: 0
employmentTypes blank but categories not: 0
same rows (masks identical): True


# Remove empty rows and empty columns

In [5]:
df1 = df[df['categories'].notna() & (df['categories'].str.strip() != '')].drop(columns=['occupationId', 'status_id'])
print(df1.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1044597 entries, 0 to 1048584
Data columns (total 20 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   categories                          1044597 non-null  object 
 1   employmentTypes                     1044597 non-null  object 
 2   metadata_expiryDate                 1044597 non-null  object 
 3   metadata_isPostedOnBehalf           1044597 non-null  bool   
 4   metadata_jobPostId                  1044597 non-null  object 
 5   metadata_newPostingDate             1044597 non-null  object 
 6   metadata_originalPostingDate        1044597 non-null  object 
 7   metadata_repostCount                1044597 non-null  int64  
 8   metadata_totalNumberJobApplication  1044597 non-null  int64  
 9   metadata_totalNumberOfView          1044597 non-null  int64  
 10  minimumYearsExperience              1044597 non-null  int64  
 11  numberOfVac

# Change title to all lower case #

# Change data type to the right ones and change title to lower case #

In [6]:
date_cols = ['metadata_expiryDate', 'metadata_newPostingDate', 'metadata_originalPostingDate']
category_cols = ['employmentTypes', 'positionLevels', 'postedCompany_name', 'salary_type', 'status_jobStatus']

df1[date_cols] = df1[date_cols].apply(pd.to_datetime)
df1[category_cols] = df1[category_cols].astype('category')

df1['metadata_jobPostId'] = df1['metadata_jobPostId'].astype('string')
df1['title'] = df1['title'].astype('string').str.lower()

print(df1.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1044597 entries, 0 to 1048584
Data columns (total 20 columns):
 #   Column                              Non-Null Count    Dtype         
---  ------                              --------------    -----         
 0   categories                          1044597 non-null  object        
 1   employmentTypes                     1044597 non-null  category      
 2   metadata_expiryDate                 1044597 non-null  datetime64[ns]
 3   metadata_isPostedOnBehalf           1044597 non-null  bool          
 4   metadata_jobPostId                  1044597 non-null  string        
 5   metadata_newPostingDate             1044597 non-null  datetime64[ns]
 6   metadata_originalPostingDate        1044597 non-null  datetime64[ns]
 7   metadata_repostCount                1044597 non-null  int64         
 8   metadata_totalNumberJobApplication  1044597 non-null  int64         
 9   metadata_totalNumberOfView          1044597 non-null  int64         

In [7]:
# Columns to check for duplicates (all except metadata_jobPostId)
cols_to_check = [col for col in df1.columns if col != "metadata_jobPostId"]

# Find rows that are duplicated based on those columns
# keep=False marks ALL occurrences (not just the 2nd, 3rd, etc.) as duplicates
duplicate_mask = df1.duplicated(subset=cols_to_check, keep=False)

duplicate_rows = df1[duplicate_mask]

# Optional: sort so duplicate groups sit next to each other for easy comparison
duplicate_rows = duplicate_rows.sort_values(by=cols_to_check)

print(f"Found {len(duplicate_rows)} duplicate rows")
# Drop duplicates, keeping the first occurrence of each group
df2 = df1.drop_duplicates(subset=cols_to_check, keep='first')
#print(f"Remaining rows after dropping duplicates: {len(df2)}")
df2.info()

Found 19709 duplicate rows
<class 'pandas.core.frame.DataFrame'>
Int64Index: 1031725 entries, 0 to 1048584
Data columns (total 20 columns):
 #   Column                              Non-Null Count    Dtype         
---  ------                              --------------    -----         
 0   categories                          1031725 non-null  object        
 1   employmentTypes                     1031725 non-null  category      
 2   metadata_expiryDate                 1031725 non-null  datetime64[ns]
 3   metadata_isPostedOnBehalf           1031725 non-null  bool          
 4   metadata_jobPostId                  1031725 non-null  string        
 5   metadata_newPostingDate             1031725 non-null  datetime64[ns]
 6   metadata_originalPostingDate        1031725 non-null  datetime64[ns]
 7   metadata_repostCount                1031725 non-null  int64         
 8   metadata_totalNumberJobApplication  1031725 non-null  int64         
 9   metadata_totalNumberOfView          10317

In [8]:
df2.to_csv('SGJobData_cleaned.csv', index=False)

# Not necessary since Streamlit can process columns with json data well? # Stop here for now

import json

df1['categories_parsed'] = df1['categories'].apply(json.loads)
df1['categories_parsed'].head().to_excel('categories_parsed check.xlsx')


df_categories = df1[['metadata_jobPostId', 'categories_parsed']].explode('categories_parsed')
df_categories = df_categories.dropna(subset=['categories_parsed'])
df_categories['category_id'] = df_categories['categories_parsed'].apply(lambda d: d['id'])
df_categories['category'] = df_categories['categories_parsed'].apply(lambda d: d['category'])
df_categories = df_categories.drop(columns='categories_parsed')

df_categories.head()

def parse_json(val):
    if isinstance(val, str):
        return json.loads(val)
    return val

records = df1['categories'].apply(parse_json).explode()
result_df = pd.json_normalize(records)[['id', 'category']]
result_df = result_df.drop_duplicates(subset='id').reset_index(drop=True)
result_df.head()

result_df = result_df.drop_duplicates(subset='id').reset_index(drop=True)
result_df



if isinstance(df1['categories'].iloc[0], str):
    df1['categories'] = df1['categories'].apply(ast.literal_eval)

# Step 2: explode so each category dict gets its own row
df_map = df1[['metadata_jobPostId', 'categories']].explode('categories').reset_index(drop=True)

# Step 3: pull out the 'id' (and optionally 'category' name) from each dict
df_map['category_id'] = df_map['categories'].apply(lambda x: x['id'])
df_map['category_name'] = df_map['categories'].apply(lambda x: x['category'])

# Step 4: drop the now-unneeded dict column
df_map = df_map.drop(columns=['categories'])
df_map